<a href="https://colab.research.google.com/github/BuruhArloji/PythonDataScienceHandbook/blob/master/Forecast_SKU_Canonical_Pipeline_V4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sales 2025 SKU Real vs Edit Visualization

Notebook ini fokus ke satu tujuan: melihat dampak penyamaan SKU terhadap trend `CF` di `Sales 2025`.

Rule alignment:
- BIG 1625 / 1.625L -> BIG 1L terbaru
- BIG 3100 / 3.1L -> BIG 3L terbaru
- BIG 400ml dengan flavour sama -> SKU 400ml terbaru
- BIG Nipis 350ml -> deskripsi terbaru, SKU tetap sama
- VOLT 200ml -> SKU/deskripsi 24-pack terbaru
- Jika SKU 12-pack diarahkan ke 24-pack, `CF Edit = CF Real / 2`

In [ ]:
# Memuat library utama dan menyiapkan opsi tampilan dataframe/chart.

# Mengimpor modul bawaan Python untuk manipulasi path, regex, dan file zip.
from pathlib import Path
import re
import zipfile

# Mengimpor library utama untuk komputasi numerik (NumPy) dan manipulasi data (Pandas).
import numpy as np
import pandas as pd

# Mencoba mengimpor Plotly untuk visualisasi interaktif.
try:
    import plotly.express as px
except ModuleNotFoundError:
    # Mengabaikan error jika Plotly tidak terinstal.
    px = None

# Mencoba mengimpor Matplotlib untuk visualisasi statis dan pemformatan sumbu (axis).
try:
    import matplotlib.pyplot as plt
    from matplotlib.ticker import FuncFormatter
except ModuleNotFoundError:
    plt = None
    FuncFormatter = None

# Mengatur tampilan Pandas agar dapat menampilkan hingga 120 kolom dan memperlebar tampilan tabel.
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

## 1. Source File

Notebook ini hanya memakai file dari Google Drive. Ubah path di bawah kalau lokasi file di Drive berbeda.

In [ ]:
# Menentukan lokasi file dashboard di Google Drive dan memastikan file Excel dapat dibaca.

# Mendefinisikan lokasi absolut file di Google Drive.
DRIVE_SALES_FILE = Path("/content/drive/MyDrive/Data Sales/Indonesia Sales Dashboard 2026.xlsm")

try:
    from google.colab import drive
    # Menghubungkan (mount) Google Drive ke Colab.
    drive.mount("/content/drive")
except ModuleNotFoundError:
    # Mengabaikan error jika dijalankan di luar Colab, dan menampilkan peringatan.
    print("Run this notebook in Colab, or mount Google Drive manually before running.")

# Menghentikan program dengan error jika file tidak ditemukan.
if not DRIVE_SALES_FILE.exists():
    raise FileNotFoundError(f"File not found: {DRIVE_SALES_FILE}")

# Membaca file Excel sebagai arsip ZIP untuk memvalidasi isinya.
with zipfile.ZipFile(DRIVE_SALES_FILE) as zf:
    # Memastikan file tersebut adalah dokumen Office yang valid (bukan file rusak/corrupt).
    if "[Content_Types].xml" not in zf.namelist():
        raise ValueError(f"Not a readable Excel workbook: {DRIVE_SALES_FILE}")

# Menetapkan file untuk digunakan dan menampilkan konfirmasi.
SALES_FILE = DRIVE_SALES_FILE
print("Using:", SALES_FILE)

## 2. Load Data

Chart hanya memakai `Sales 2025`. `Sales 2026` dipakai sebagai acuan SKU terbaru.

In [ ]:
# Membaca sheet Sales 2025 sebagai data utama dan Sales 2026 sebagai acuan SKU terbaru.

sales25 = pd.read_excel(SALES_FILE, sheet_name="Sales 2025", engine="openpyxl")
sales26 = pd.read_excel(SALES_FILE, sheet_name="Sales 2026", engine="openpyxl")

print("Sales 2025:", sales25.shape)
print("Sales 2026 reference:", sales26.shape)
display(sales25.head(3))

## 3. Clean Real SKU

In [ ]:
# Membersihkan kolom SKU real, format, flavor, dan membuat Group Key untuk proses alignment.

def clean_text(value):
    # Mengubah teks menjadi huruf kapital dan menghapus spasi ganda atau berlebih.
    if pd.isna(value):
        return ""
    value = str(value).strip().upper()
    return re.sub(r"\s+", " ", value)


def clean_code(value):
    # Membersihkan format kode, khususnya menghapus akhiran '.0' jika terbaca sebagai float.
    if pd.isna(value):
        return ""
    value = str(value).strip()
    return value[:-2] if value.endswith(".0") else value


def group_key(brand, flavor, fmt):
    # Mengelompokkan SKU ke dalam kategori spesifik berdasarkan Brand, Flavor, dan Format (ukuran).
    brand = clean_text(brand)
    flavor = clean_text(flavor)
    if pd.isna(fmt):
        return ""
    fmt = float(fmt)

    # Menggunakan np.isclose untuk mengatasi masalah presisi angka desimal (float) saat membandingkan ukuran.
    if brand == "BIG" and "NIPIS" in flavor and np.isclose(fmt, 0.35):
        return "BIG_NIPIS_350"
    if brand == "BIG" and np.isclose(fmt, 0.4):
        return "BIG_400"
    if brand == "BIG" and (np.isclose(fmt, 1.625) or np.isclose(fmt, 1.0)):
        return "BIG_1L"
    if brand == "BIG" and (np.isclose(fmt, 3.1) or np.isclose(fmt, 3.0)):
        return "BIG_3L"
    if brand == "VOLT" and np.isclose(fmt, 0.2):
        return "VOLT_200_24"
    return ""


# Membuat DataFrame 'actual' baru dengan menerapkan fungsi pembersihan pada kolom-kolom yang relevan.
actual = pd.DataFrame({
    "Date": pd.to_datetime(sales25["Date"], errors="coerce"),
    "Channel Group": sales25["Channel Group"],
    "Branch": sales25["Branch"],
    "Channel": sales25["Channel"],
    "Cust Code": sales25["Cust Code"].map(clean_code),
    "Customer Name": sales25["Customer Name"],
    "Brand": sales25["Brand"].map(clean_text),
    "Flavor": sales25["Flavor"].map(clean_text),
    "Format": pd.to_numeric(sales25["Format"], errors="coerce"),
    "Box Content": sales25["Box Content"].map(clean_code),
    "CF": pd.to_numeric(sales25["CF"], errors="coerce").fillna(0),
    "Item Code Real": sales25["Item Code (Real)"].map(clean_code),
    "Short Item Description Real": sales25["Short Item Description (Edit).1"].map(clean_text),
})

# Menghapus baris yang tidak memiliki data tanggal (invalid date).
actual = actual.dropna(subset=["Date"]).copy()

# Membuat kolom 'Month' yang berisi tanggal awal dari bulan transaksi tersebut.
actual["Month"] = actual["Date"].dt.to_period("M").dt.to_timestamp()

# Menerapkan fungsi group_key ke seluruh baris data menggunakan list comprehension.
actual["Group Key"] = [group_key(b, f, s) for b, f, s in zip(actual["Brand"], actual["Flavor"], actual["Format"])]

# Membuat kolom 'SKU Real' dengan menggabungkan kode item dan deskripsi sebagai identifier unik.
actual["SKU Real"] = actual["Item Code Real"] + " | " + actual["Short Item Description Real"]

# Menampilkan 10 baris pertama dari DataFrame yang sudah dibersihkan.
display(actual.head(10))

## 4. Build Latest SKU Reference

Reference diambil dari `Sales 2026`, lalu dipilih row terbaru per Brand + Flavor + Group Key. Untuk VOLT, reference dibatasi ke `Box Content = 24`.

### 4.1. Upload and Cleaning Sales 2026

In [ ]:
# Mengambil dan membersihkan data mentah dari sheet Sales 2026.

ref = pd.DataFrame({
    # Mengonversi tanggal, baris error akan menjadi NaT.
    "Reference Date": pd.to_datetime(sales26["fecha_liquidacion"], errors="coerce"),
    "Brand": sales26["desc_marca"].map(clean_text),
    "Flavor": sales26["desc_sabor"].map(clean_text),
    "Format": pd.to_numeric(sales26["desc_formato"], errors="coerce"),
    "Box Content": sales26["cant_contenido"].map(clean_code),
    "CF": pd.to_numeric(sales26["CF"], errors="coerce").fillna(0),
    "Item Code Edit": sales26["cod_articulo"].map(clean_code),
    "Short Item Description Edit": sales26["desc_articulo_corto"].map(clean_text),
})

# Menghapus baris yang tidak memiliki tanggal referensi valid.
ref = ref.dropna(subset=["Reference Date"]).copy()

In [ ]:
# Membuat kunci pengelompokan dan membuang data yang tidak sesuai kriteria.

# Menerapkan fungsi group_key untuk standarisasi kategori produk.
ref["Group Key"] = [group_key(b, f, s) for b, f, s in zip(ref["Brand"], ref["Flavor"], ref["Format"])]

# Membuang data yang tidak masuk ke dalam kategori Group Key yang sudah ditentukan.
ref = ref[ref["Group Key"] != ""].copy()

# Filter khusus: Jika Group Key adalah "VOLT_200_24", pastikan isi box-nya adalah 24.
ref = ref[(ref["Group Key"] != "VOLT_200_24") | (ref["Box Content"] == "24")]

In [ ]:
# Menyusun tabel acuan akhir yang berisi SKU versi paling baru.

sku_reference = (
    # Mengurutkan data. Reference Date dan CF diurutkan menurun (False) agar data terbaru berada di atas.
    ref.sort_values(["Brand", "Flavor", "Group Key", "Reference Date", "CF"], ascending=[True, True, True, False, False])

    # Menghapus duplikat berdasarkan Brand, Flavor, dan Group Key.
    # Karena data terbaru ada di atas (berkat sort_values), keep='first' (default) akan menyimpan SKU yang paling update.
    .drop_duplicates(["Brand", "Flavor", "Group Key"])

    # Memilih hanya kolom-kolom yang diperlukan untuk tabel referensi.
    [[
        "Brand", "Flavor", "Group Key", "Reference Date",
        "Item Code Edit", "Short Item Description Edit", "Format", "Box Content",
    ]]

    # Mengganti nama kolom agar tidak tertukar dengan data aktual (Sales 2025).
    .rename(columns={"Format": "Reference Format", "Box Content": "Reference Box Content"})
)

# Menampilkan hasil tabel referensi SKU yang sudah jadi.
display(sku_reference.sort_values(["Brand", "Group Key", "Flavor"]))

## 5. Apply SKU Edit

In [ ]:
# LEFT JOIN data aktual dan acuan SKU berdasarkan Brand, Flavor, dan Group Key.
aligned = actual.merge(sku_reference, on=["Brand", "Flavor", "Group Key"], how="left")

# Mengecek keberadaan referensi yang valid.
has_reference = aligned["Group Key"].ne("") & aligned["Item Code Edit"].notna()

In [ ]:
# Gunakan data edit terbaru jika ada referensi, jika tidak gunakan data aktual.
aligned["Item Code Edit"] = np.where(has_reference, aligned["Item Code Edit"], aligned["Item Code Real"])

aligned["Short Item Description Edit"] = np.where(
    has_reference,
    aligned["Short Item Description Edit"],
    aligned["Short Item Description Real"],
)

In [ ]:
# ==========================================
# 5. 3. FORMAT SKU EDIT & DETEKSI PERUBAHAN
# ==========================================
# Buat format identifier gabungan 'KODE | DESKRIPSI'.
aligned["SKU Edit"] = aligned["Item Code Edit"].map(clean_code) + " | " + aligned["Short Item Description Edit"].map(clean_text)

# Tandai baris yang mengalami perubahan SKU.
aligned["Real != Edit"] = aligned["SKU Real"] != aligned["SKU Edit"]

In [ ]:
# ==========================================
# 5. 4. PENYESUAIAN CASE FACTOR (CF)
# ==========================================
aligned["CF Real"] = aligned["CF"]

# Deteksi perubahan pack dari 12 ke 24.
pack_12_to_24 = aligned["Box Content"].eq("12") & aligned["Reference Box Content"].eq("24") & aligned["Real != Edit"]

# Bagi nilai CF menjadi dua jika terjadi perubahan pack.
aligned["CF Edit"] = np.where(pack_12_to_24, aligned["CF Real"] / 2, aligned["CF Real"])
aligned["CF Adjustment"] = np.where(pack_12_to_24, "12-pack to 24-pack: CF / 2", "No CF adjustment")

In [ ]:

# ==========================================
# 5. 5. SELEKSI KOLOM & METADATA WAKTU
# ==========================================
# Mengambil kolom final yang dibutuhkan untuk visualisasi, append 2026, dan forecast.
sales_2025_visual = aligned[[
    "Date", "Channel Group", "Branch", "Channel", "Cust Code", "Customer Name",
    "Short Item Description Real", "Item Code Real",
    "Short Item Description Edit", "Item Code Edit",
    "Format", "Box Content", "Reference Format", "Reference Box Content",
    "CF Real", "CF Edit", "CF Adjustment",
    "Group Key", "SKU Real", "SKU Edit", "Real != Edit",
]].copy()

# Menambahkan identitas sheet sumber dan hierarki tanggal.
sales_2025_visual.insert(0, "Source Sheet", "Sales 2025")
sales_2025_visual.insert(1, "Year", sales_2025_visual["Date"].dt.year)
sales_2025_visual.insert(2, "Month No", sales_2025_visual["Date"].dt.month)
sales_2025_visual["Month"] = sales_2025_visual["Date"].dt.to_period("M").dt.to_timestamp()

# ==========================================
# 5. 6. RINGKASAN OUTPUT & TAMPILAN DATA
# ==========================================
print("Rows:", len(sales_2025_visual))
print("Rows changed Real -> Edit:", int(sales_2025_visual["Real != Edit"].sum()))
print("Rows with CF / 2 adjustment:", int((sales_2025_visual["CF Adjustment"] == "12-pack to 24-pack: CF / 2").sum()))

display(sales_2025_visual.head(10))


## 6. Mapping Summary

In [ ]:
# ==========================================
# 7. 6. MAPPING SUMMARY
# ==========================================
# Meringkas pasangan SKU Real ke SKU Edit agar perubahan alignment mudah diaudit.

mapping_summary = (
    sales_2025_visual[sales_2025_visual["Real != Edit"]]
    .groupby(["Group Key", "SKU Real", "SKU Edit"], as_index=False)
    .agg(
        rows=("CF Real", "size"),
        cf_real=("CF Real", "sum"),
        cf_edit=("CF Edit", "sum"),
        cf_adjustment=("CF Adjustment", lambda s: ", ".join(sorted(set(s)))),
        first_date=("Date", "min"),
        last_date=("Date", "max"),
    )
    .sort_values(["Group Key", "SKU Real"])
)

display(mapping_summary)

## 7. Sales 2026 Alignment

In [ ]:
# ==========================================
# 6. 1. SALES 2026 DATA PREPARATION
# ==========================================
actual26 = pd.DataFrame({
    "Date": pd.to_datetime(sales26["fecha_liquidacion"], errors="coerce"),
    "Channel Group": sales26["DescCanalLocal (grupo)"],
    "Branch": sales26["desc_sucursal"],
    "Channel": sales26["DescCanalLocal"],
    "Cust Code": sales26["cod_cliente"].map(clean_code),
    "Customer Name": sales26["nomb_cliente"],
    "Brand": sales26["desc_marca"].map(clean_text),
    "Flavor": sales26["desc_sabor"].map(clean_text),
    "Format": pd.to_numeric(sales26["desc_formato"], errors="coerce"),
    "Box Content": sales26["cant_contenido"].map(clean_code),
    "CF": pd.to_numeric(sales26["CF"], errors="coerce").fillna(0),
    "Item Code Real": sales26["cod_articulo"].map(clean_code),
    "Short Item Description Real": sales26["desc_articulo_corto"].map(clean_text),
})

actual26 = actual26.dropna(subset=["Date"]).copy()
actual26["Month"] = actual26["Date"].dt.to_period("M").dt.to_timestamp()

actual26["Group Key"] = [group_key(b, f, s) for b, f, s in zip(actual26["Brand"], actual26["Flavor"], actual26["Format"])]
actual26["SKU Real"] = actual26["Item Code Real"] + " | " + actual26["Short Item Description Real"]

In [ ]:
# ==========================================
# 6. 2. ALIGNMENT & SKU UPDATES
# ==========================================
aligned26 = actual26.merge(sku_reference, on=["Brand", "Flavor", "Group Key"], how="left")
has_reference26 = aligned26["Group Key"].ne("") & aligned26["Item Code Edit"].notna()

aligned26["Item Code Edit"] = np.where(has_reference26, aligned26["Item Code Edit"], aligned26["Item Code Real"])
aligned26["Short Item Description Edit"] = np.where(
    has_reference26,
    aligned26["Short Item Description Edit"],
    aligned26["Short Item Description Real"],
)

aligned26["SKU Edit"] = aligned26["Item Code Edit"].map(clean_code) + " | " + aligned26["Short Item Description Edit"].map(clean_text)
aligned26["Real != Edit"] = aligned26["SKU Real"] != aligned26["SKU Edit"]

In [ ]:
# ==========================================
# 6. 3. CF ADJUSTMENT
# ==========================================
aligned26["CF Real"] = aligned26["CF"]
pack_12_to_24_26 = aligned26["Box Content"].eq("12") & aligned26["Reference Box Content"].eq("24") & aligned26["Real != Edit"]

aligned26["CF Edit"] = np.where(pack_12_to_24_26, aligned26["CF Real"] / 2, aligned26["CF Real"])
aligned26["CF Adjustment"] = np.where(pack_12_to_24_26, "12-pack to 24-pack: CF / 2", "No CF adjustment")

In [ ]:
# ==========================================
# 6. 4. COLUMN SELECTION & METADATA
# ==========================================
sales_2026_visual = aligned26[[
    "Date", "Channel Group", "Branch", "Channel", "Cust Code", "Customer Name",
    "Short Item Description Real", "Item Code Real",
    "Short Item Description Edit", "Item Code Edit",
    "Format", "Box Content", "Reference Format", "Reference Box Content",
    "CF Real", "CF Edit", "CF Adjustment",
    "Group Key", "SKU Real", "SKU Edit", "Real != Edit",
]].copy()

sales_2026_visual.insert(0, "Source Sheet", "Sales 2026")
sales_2026_visual.insert(1, "Year", sales_2026_visual["Date"].dt.year)
sales_2026_visual.insert(2, "Month No", sales_2026_visual["Date"].dt.month)
sales_2026_visual["Month"] = sales_2026_visual["Date"].dt.to_period("M").dt.to_timestamp()

In [ ]:
# ==========================================
# 6. 5. MAPPING SUMMARY & OUTPUT
# ==========================================
mapping_summary_2026 = (
    sales_2026_visual[sales_2026_visual["Real != Edit"]]
    .groupby(["Group Key", "SKU Real", "SKU Edit"], as_index=False)
    .agg(
        rows=("CF Real", "size"),
        cf_real=("CF Real", "sum"),
        cf_edit=("CF Edit", "sum"),
        cf_adjustment=("CF Adjustment", lambda s: ", ".join(sorted(set(s)))),
        first_date=("Date", "min"),
        last_date=("Date", "max"),
    )
    .sort_values(["Group Key", "SKU Real"])
)

print("Sales 2026 rows:", len(sales_2026_visual))
print("Sales 2026 rows changed Real -> Edit:", int(sales_2026_visual["Real != Edit"].sum()))
print("Sales 2026 rows with CF / 2 adjustment:", int((sales_2026_visual["CF Adjustment"] == "12-pack to 24-pack: CF / 2").sum()))

display(mapping_summary_2026)

## 8. Append Sales 2025 + Sales 2026

In [ ]:
# ==========================================
# 9. 8. APPEND SALES 2025 + SALES 2026
# ==========================================
# Menggabungkan data aligned 2025 dan 2026 menjadi satu dataset analisis.

append_columns = [
    "Source Sheet", "Year", "Month No", "Date",
    "Channel Group", "Branch", "Channel", "Cust Code", "Customer Name",
    "Short Item Description Real", "Item Code Real",
    "Short Item Description Edit", "Item Code Edit",
    "Format", "Box Content", "Reference Format", "Reference Box Content",
    "CF Real", "CF Edit", "CF Adjustment",
    "Group Key", "SKU Real", "SKU Edit", "Real != Edit",
]

sales_aligned_append = pd.concat(
    [
        sales_2025_visual[append_columns],
        sales_2026_visual[append_columns],
    ],
    ignore_index=True,
)
sales_aligned_append["Month"] = sales_aligned_append["Date"].dt.to_period("M").dt.to_timestamp()

print("Appended rows:", len(sales_aligned_append))
print("Sales 2025 rows:", len(sales_2025_visual))
print("Sales 2026 rows:", len(sales_2026_visual))
print("Total rows changed Real -> Edit:", int(sales_aligned_append["Real != Edit"].sum()))
display(sales_aligned_append.head(10))

## 9. VOLT CF / 2 Check - Real vs Edit

In [ ]:
# ==========================================
# 10. 9. VOLT CF / 2 CHECK - REAL VS EDIT
# ==========================================
# Memvisualisasikan VOLT sebelum dan sesudah CF dibagi 2 untuk audit pack 12 ke 24.

keyword = "VOLT"

filtered = sales_aligned_append[
    sales_aligned_append["SKU Real"].str.contains(keyword, case=False, na=False)
    | sales_aligned_append["SKU Edit"].str.contains(keyword, case=False, na=False)
].copy()

filtered_long = pd.concat(
    [
        filtered.assign(SKU_View="Real Before CF / 2", SKU=filtered["SKU Real"], CF_View=filtered["CF Real"]),
        filtered.assign(SKU_View="Edit After CF / 2", SKU=filtered["SKU Edit"], CF_View=filtered["CF Edit"]),
    ],
    ignore_index=True,
)

filtered_monthly = (
    filtered_long
    .groupby(["Month", "SKU_View", "SKU"], as_index=False)
    .agg(CF=("CF_View", "sum"))
)

display(filtered_monthly)

if px:
    fig = px.line(
        filtered_monthly,
        x="Month",
        y="CF",
        color="SKU",
        line_dash="SKU_View",
        markers=True,
        category_orders={"SKU_View": ["Real Before CF / 2", "Edit After CF / 2"]},
        title=f"{keyword}: Real Before CF / 2 vs Edit After CF / 2",
    )
    fig.show()

## 10. Appended Monthly CF by SKU - Edit

In [ ]:
# ==========================================
# 11. 10. APPENDED MONTHLY CF BY SKU - EDIT
# ==========================================
# Membuat line chart bulanan berdasarkan SKU Edit setelah data 2025 dan 2026 digabung.

append_edit_monthly = sales_aligned_append.groupby(["Month", "SKU Edit"], as_index=False).agg(CF=("CF Edit", "sum"))
top_append_edit = append_edit_monthly.groupby("SKU Edit")["CF"].sum().nlargest(20).index
append_edit_top = append_edit_monthly[append_edit_monthly["SKU Edit"].isin(top_append_edit)]

if px:
    fig = px.line(append_edit_top, x="Month", y="CF", color="SKU Edit", markers=True, title="Sales 2025-2026 Monthly CF by SKU - Edit Top 20")
    fig.show()
else:
    display(append_edit_top.head(50))

## 11. Sales 2026 Monthly CF by SKU - Edit

In [ ]:
# ==========================================
# 12. 11. SALES 2026 MONTHLY CF BY SKU - EDIT
# ==========================================
# Mengecek tren 2026 saja setelah alignment SKU.

edit_monthly_2026 = sales_2026_visual.groupby(["Month", "SKU Edit"], as_index=False).agg(CF=("CF Edit", "sum"))
top_edit_2026 = edit_monthly_2026.groupby("SKU Edit")["CF"].sum().nlargest(20).index
edit_top_2026 = edit_monthly_2026[edit_monthly_2026["SKU Edit"].isin(top_edit_2026)]

if px:
    fig = px.line(edit_top_2026, x="Month", y="CF", color="SKU Edit", markers=True, title="Sales 2026 Monthly CF by SKU - Edit Top 20")
    fig.show()
else:
    display(edit_top_2026.head(50))

## 12. Distinct SKU Count: Real vs Edit

In [ ]:
# ==========================================
# 13. 12. DISTINCT SKU COUNT: REAL VS EDIT
# ==========================================
# Membandingkan jumlah SKU unik sebelum dan sesudah alignment per bulan.

distinct_monthly = (
    sales_aligned_append.groupby("Month")
    .agg(
        real_sku_count=("SKU Real", "nunique"),
        edit_sku_count=("SKU Edit", "nunique"),
        cf_real=("CF Real", "sum"),
        cf_edit=("CF Edit", "sum"),
    )
    .reset_index()
)

display(distinct_monthly)

if px:
    fig = px.line(
        distinct_monthly,
        x="Month",
        y=["real_sku_count", "edit_sku_count"],
        markers=True,
        title="Distinct SKU Count per Month: Real vs Edit - Sales 2025-2026",
    )
    fig.show()

## 13. Monthly CF by SKU - Real

In [ ]:
# ==========================================
# 14. 13. MONTHLY CF BY SKU - REAL
# ==========================================
# Menampilkan trend bulanan berdasarkan SKU Real sebagai pembanding sebelum edit.

real_monthly = sales_aligned_append.groupby(["Month", "SKU Real"], as_index=False).agg(CF=("CF Real", "sum"))
top_real = real_monthly.groupby("SKU Real")["CF"].sum().nlargest(20).index
real_top = real_monthly[real_monthly["SKU Real"].isin(top_real)]

if px:
    fig = px.line(real_top, x="Month", y="CF", color="SKU Real", markers=True, title="Sales 2025-2026 Monthly CF by SKU - Real Top 20")
    fig.show()
else:
    display(real_top.head(50))

## 14. Monthly CF by SKU - Edit

In [ ]:
# ==========================================
# 15. 14. MONTHLY CF BY SKU - EDIT
# ==========================================
# Menampilkan trend bulanan berdasarkan SKU Edit sebagai hasil setelah alignment.

edit_monthly = sales_aligned_append.groupby(["Month", "SKU Edit"], as_index=False).agg(CF=("CF Edit", "sum"))
top_edit = edit_monthly.groupby("SKU Edit")["CF"].sum().nlargest(20).index
edit_top = edit_monthly[edit_monthly["SKU Edit"].isin(top_edit)]

if px:
    fig = px.line(edit_top, x="Month", y="CF", color="SKU Edit", markers=True, title="Sales 2025-2026 Monthly CF by SKU - Edit Top 20")
    fig.show()
else:
    display(edit_top.head(50))

## 15. Sep-Nov 2026 Forecast Detail

Forecast ini memakai grain `Cust Code x SKU`, dengan baseline yang dibersihkan dari suspected promo/outlier secara konservatif.
Sebelum forecast, `Customer Name` dan `Branch` diselaraskan dulu agar 1 Cust Code hanya punya 1 nama dan 1 branch utama.

Parameter utama:
- `FORECAST_MONTH`: bulan yang akan diforecast
- `RUNRATE_MONTH`: bulan berjalan yang masih MTD
- `CF Edit`: volume yang sudah disetarakan SKU dan pack
- Forecast engine difilter hanya untuk `Channel Group = MODERN`
- Jika 1 Cust Code muncul dengan beberapa nama atau branch, pakai nama dan branch dengan jumlah transaksi/order row terbanyak
- Agustus run-rate memakai multiplier konservatif `1.5`
- Juli dianggap promo dan tidak menjadi driver utama baseline September


In [ ]:
# ==========================================
# 15. SEP-NOV 2026 FORECAST DETAIL
# ==========================================
# Menyiapkan data forecast MODERN, parameter run-rate, seasonal reference, dan grain Customer x SKU.

# Menetapkan bulan forecast, bulan run-rate, multiplier run-rate, dan scope channel
FORECAST_MONTHS = pd.date_range("2026-09-01", "2026-11-01", freq="MS")
FORECAST_MONTH = FORECAST_MONTHS[0]
RUNRATE_MONTH = FORECAST_MONTH - pd.DateOffset(months=1)
RUNRATE_MULTIPLIER = 1.5
FORECAST_CHANNEL_GROUP = "MODERN"
KNOWN_PROMO_MONTHS = [pd.Timestamp("2026-07-01")]

# Parameter seasonal index 2025
SEASONAL_REFERENCE_YEAR = 2025
SEASONAL_BASE_MONTHS = pd.date_range("2025-03-01", "2025-08-01", freq="MS")
SEASONAL_BLEND_WEIGHT = 0.30
SEASONAL_MIN_INDEX = 0.80
SEASONAL_MAX_INDEX = 1.25

# Grain forecast: Cust Code x SKU
key_cols = [
    "Channel Group", "Branch", "Cust Code", "Customer Name",
    "Item Code Edit", "Short Item Description Edit", "SKU Edit",
]

# Filter forecast hanya untuk Channel Group MODERN
forecast_source = sales_aligned_append[
    sales_aligned_append["Channel Group"].map(clean_text).eq(FORECAST_CHANNEL_GROUP)
].copy()

# ==========================================
# 0. BRANCH OVERRIDE (BALI & PEKANBARU)
# ==========================================
# Memastikan customer area ekspansi terpetakan ke branch baru untuk konsolidasi volume historis
area_ref_2026 = sales_aligned_append[
    (sales_aligned_append['Year'] >= 2026) &
    (sales_aligned_append['Branch'].str.contains('BALI|PEKANBARU', case=False, na=False))
].copy()

bali_cust_exp = area_ref_2026[area_ref_2026['Branch'].str.contains('BALI', case=False, na=False)]['Cust Code'].unique()
pku_cust_exp = area_ref_2026[area_ref_2026['Branch'].str.contains('PEKANBARU', case=False, na=False)]['Cust Code'].unique()

forecast_source.loc[forecast_source['Cust Code'].isin(bali_cust_exp), 'Branch'] = 'LOGISTIC PROVIDER BALI'
forecast_source.loc[forecast_source['Cust Code'].isin(pku_cust_exp), 'Branch'] = 'LOGISTIC PROVIDER PEKANBARU'

# ==========================================
# 1. NORMALISASI IDENTITAS CUSTOMER
# ==========================================
# Simpan identitas asli sebelum normalisasi untuk audit/export CSV
forecast_source["Original Customer Name"] = forecast_source["Customer Name"]
forecast_source["Original Branch"] = forecast_source["Branch"]

customer_name_reference = (
    forecast_source
    .groupby(["Cust Code", "Customer Name"], as_index=False)
    .agg(order_rows=("CF Edit", "size"), cf_edit=("CF Edit", "sum"), last_order_date=("Date", "max"))
    .sort_values(["Cust Code", "order_rows", "cf_edit"], ascending=[True, False, False])
    .drop_duplicates(["Cust Code"], keep="first")
    .rename(columns={"Customer Name": "Primary Customer Name"})
)

customer_branch_reference = (
    forecast_source[forecast_source["Year"] >= 2026]
    .groupby(["Cust Code", "Branch"], as_index=False)
    .agg(order_rows=("CF Edit", "size"), cf_edit=("CF Edit", "sum"), last_order_date=("Date", "max"))
    .sort_values(["Cust Code", "order_rows", "cf_edit"], ascending=[True, False, False])
    .drop_duplicates(["Cust Code"], keep="first")
    .rename(columns={"Branch": "Primary Branch"})
)

forecast_source = forecast_source.merge(customer_name_reference[["Cust Code", "Primary Customer Name"]], on="Cust Code", how="left")
forecast_source = forecast_source.merge(customer_branch_reference[["Cust Code", "Primary Branch"]], on="Cust Code", how="left")
forecast_source["Customer Name"] = forecast_source["Primary Customer Name"].fillna(forecast_source["Customer Name"])
forecast_source["Branch"] = forecast_source["Primary Branch"].fillna(forecast_source["Branch"])
# Kolom Primary dihapus tetapi Original tetap ada
forecast_source = forecast_source.drop(columns=["Primary Customer Name", "Primary Branch"])

# ==========================================
# 2. UC FACTOR & AGREGASI BULANAN
# ==========================================
forecast_source["Format Edit"] = pd.to_numeric(forecast_source["Reference Format"], errors="coerce").fillna(pd.to_numeric(forecast_source["Format"], errors="coerce"))
forecast_source["Box Content Edit Numeric"] = pd.to_numeric(forecast_source["Reference Box Content"], errors="coerce").fillna(pd.to_numeric(forecast_source["Box Content"], errors="coerce"))
forecast_source["UC Factor"] = forecast_source["Format Edit"] * forecast_source["Box Content Edit Numeric"] / 30

uc_reference = forecast_source.groupby(key_cols, as_index=False).agg(UC_Factor=("UC Factor", "max"))

monthly_customer_sku = (
    forecast_source
    .groupby(["Month", *key_cols], as_index=False)
    .agg(CF=("CF Edit", "sum"))
)

print(f"Forecast Source Rows: {len(forecast_source)}")
print(f"Bali Customers Mapped: {len(bali_cust_exp)}")
print(f"Pekanbaru Customers Mapped: {len(pku_cust_exp)}")
display(monthly_customer_sku.head())


In [ ]:
# ==========================================
# 15. SEP 2026 FORECAST DETAIL
# ==========================================
# Parameter dan data source untuk forecast MODERN.

FORECAST_MONTH = pd.Timestamp("2026-09-01")
RUNRATE_MONTH = FORECAST_MONTH - pd.DateOffset(months=1)
RUNRATE_MULTIPLIER = 1.5
FORECAST_CHANNEL_GROUP = "MODERN"
KNOWN_PROMO_MONTHS = [pd.Timestamp("2026-07-01")]

# Grain forecast: Customer x SKU
key_cols = [
    "Channel Group", "Branch", "Cust Code", "Customer Name",
    "Item Code Edit", "Short Item Description Edit", "SKU Edit",
]

# Filter data hanya untuk channel MODERN
forecast_source = sales_aligned_append[
    sales_aligned_append["Channel Group"].map(clean_text).eq(FORECAST_CHANNEL_GROUP)
].copy()

In [ ]:
# ==========================================
# 0. BRANCH OVERRIDE (BALI & PEKANBARU)
# ==========================================
# Memetakan customer area ekspansi ke branch baru untuk konsolidasi.

area_ref_2026 = sales_aligned_append[
    (sales_aligned_append['Year'] >= 2026) &
    (sales_aligned_append['Branch'].str.contains('BALI|PEKANBARU', case=False, na=False))
].copy()

bali_cust_exp = area_ref_2026[area_ref_2026['Branch'].str.contains('BALI', case=False, na=False)]['Cust Code'].unique()
pku_cust_exp = area_ref_2026[area_ref_2026['Branch'].str.contains('PEKANBARU', case=False, na=False)]['Cust Code'].unique()

forecast_source.loc[forecast_source['Cust Code'].isin(bali_cust_exp), 'Branch'] = 'LOGISTIC PROVIDER BALI'
forecast_source.loc[forecast_source['Cust Code'].isin(pku_cust_exp), 'Branch'] = 'LOGISTIC PROVIDER PEKANBARU'

In [ ]:
# ==========================================
# 1. NORMALISASI IDENTITAS CUSTOMER
# ==========================================
# Menyeragamkan nama customer dan branch menggunakan data transaksi terbanyak/terakhir.

# Menyimpan identitas asli sebelum ditimpa
forecast_source["Original Customer Name"] = forecast_source["Customer Name"]
forecast_source["Original Branch"] = forecast_source["Branch"]

customer_name_reference = (
    forecast_source
    .groupby(["Cust Code", "Customer Name"], as_index=False)
    .agg(order_rows=("CF Edit", "size"), cf_edit=("CF Edit", "sum"), last_order_date=("Date", "max"))
    .sort_values(["Cust Code", "order_rows", "cf_edit"], ascending=[True, False, False])
    .drop_duplicates(["Cust Code"], keep="first")
    .rename(columns={"Customer Name": "Primary Customer Name"})
)

customer_branch_reference = (
    forecast_source[forecast_source["Year"] >= 2026]
    .groupby(["Cust Code", "Branch"], as_index=False)
    .agg(order_rows=("CF Edit", "size"), cf_edit=("CF Edit", "sum"), last_order_date=("Date", "max"))
    .sort_values(["Cust Code", "order_rows", "cf_edit"], ascending=[True, False, False])
    .drop_duplicates(["Cust Code"], keep="first")
    .rename(columns={"Branch": "Primary Branch"})
)

forecast_source = forecast_source.merge(customer_name_reference[["Cust Code", "Primary Customer Name"]], on="Cust Code", how="left")
forecast_source = forecast_source.merge(customer_branch_reference[["Cust Code", "Primary Branch"]], on="Cust Code", how="left")

# Menerapkan hasil normalisasi
forecast_source["Customer Name"] = forecast_source["Primary Customer Name"].fillna(forecast_source["Customer Name"])
forecast_source["Branch"] = forecast_source["Primary Branch"].fillna(forecast_source["Branch"])
forecast_source = forecast_source.drop(columns=["Primary Customer Name", "Primary Branch"])

In [ ]:
# ==========================================
# 2. UC FACTOR & AGREGASI BULANAN
# ==========================================
# Menghitung Unit Case (UC) factor dan melakukan agregasi bulanan.

forecast_source["Format Edit"] = pd.to_numeric(forecast_source["Reference Format"], errors="coerce").fillna(pd.to_numeric(forecast_source["Format"], errors="coerce"))
forecast_source["Box Content Edit Numeric"] = pd.to_numeric(forecast_source["Reference Box Content"], errors="coerce").fillna(pd.to_numeric(forecast_source["Box Content"], errors="coerce"))
forecast_source["UC Factor"] = forecast_source["Format Edit"] * forecast_source["Box Content Edit Numeric"] / 30

uc_reference = forecast_source.groupby(key_cols, as_index=False).agg(UC_Factor=("UC Factor", "max"))

monthly_customer_sku = (
    forecast_source
    .groupby(["Month", *key_cols], as_index=False)
    .agg(CF=("CF Edit", "sum"))
)

print(f"Forecast Source Rows: {len(forecast_source)}")
print(f"Bali Customers Mapped: {len(bali_cust_exp)}")
print(f"Pekanbaru Customers Mapped: {len(pku_cust_exp)}")
display(monthly_customer_sku.head())

## 16. Conservative Promo/Outlier Cleaning

In [ ]:
# ==========================================
# 17. 1. DATA PIVOTING & DATE PREPARATION
# ==========================================
# Menyiapkan rentang waktu dan mengubah data menjadi format lebar (wide).

forecast_input_months = pd.date_range("2026-03-01", RUNRATE_MONTH, freq="MS")
history_months = pd.date_range("2026-01-01", RUNRATE_MONTH, freq="MS")

# Membentuk satu baris per Customer x SKU, dengan bulan sebagai kolom.
wide = monthly_customer_sku.pivot_table(
    index=key_cols,
    columns="Month",
    values="CF",
    aggfunc="sum",
    fill_value=0,
).reset_index()

# Memastikan semua bulan histori tersedia di kolom (termasuk jika nilainya 0).
for month in history_months:
    if month not in wide.columns:
        wide[month] = 0

wide = wide[key_cols + list(history_months)]

In [ ]:
# ==========================================
# 17. 2. OUTLIER & PROMO CLEANING LOGIC
# ==========================================
# Fungsi untuk membersihkan anomali promo atau outlier per baris.

def clean_months(row):
    values = {m: float(row[m]) for m in history_months}
    clean = {}
    promo_flags = {}
    promo_uplift = {}

    for i, month in enumerate(history_months):
        actual = values[month]
        # Mengambil histori valid (nilai > 0) maksimal 3 bulan sebelumnya.
        previous = [clean[m] for m in history_months[max(0, i - 3):i] if clean.get(m, 0) > 0]

        if month in KNOWN_PROMO_MONTHS and actual > 0:
            # Batasi baseline dengan median bulan sebelumnya jika masuk bulan promo.
            if previous:
                med = float(np.median(previous))
                is_promo = True
                clean_value = min(actual, med)
            else:
                is_promo = True
                clean_value = actual
        elif len(previous) >= 2:
            # Deteksi outlier dengan batas 2x median atau median + 2x standar deviasi.
            med = float(np.median(previous))
            std = float(np.std(previous))
            threshold = max(med * 2.0, med + 2.0 * std)
            is_promo = med > 0 and actual >= 10 and actual > threshold
            clean_value = min(actual, med) if is_promo else actual
        else:
            is_promo = False
            clean_value = actual

        clean[month] = clean_value
        promo_flags[month] = is_promo
        promo_uplift[month] = max(0, actual - clean_value)

    # Kalkulasi run-rate Agustus menggunakan RUNRATE_MULTIPLIER.
    aug_actual = values.get(RUNRATE_MONTH, 0)
    aug_clean_mtd = clean.get(RUNRATE_MONTH, 0)
    aug_rr = aug_actual * RUNRATE_MULTIPLIER
    aug_clean_rr = aug_clean_mtd * RUNRATE_MULTIPLIER

    # Mengembalikan Series berisi nilai aktual, bersih, run-rate, dan detail anomali.
    return pd.Series({
        **{f"{m.strftime('%b')}_Actual": values[m] for m in forecast_input_months},
        **{f"{m.strftime('%b')}_Clean": (aug_clean_rr if m == RUNRATE_MONTH else clean[m]) for m in forecast_input_months},
        "Aug_MTD": aug_actual,
        "Aug_RunRate": aug_rr,
        "Aug_Clean_RunRate": aug_clean_rr,
        "Promo_Suspect_Months": int(sum(promo_flags.values())),
        "Promo_Suspect_Uplift": float(sum(promo_uplift.values())),
    })

In [ ]:
# ==========================================
# 17. 3. APPLY CLEANING & OUTPUT
# ==========================================
# Menerapkan fungsi ke setiap baris dan menggabungkan dengan kolom identitas awal.

clean_features = wide.apply(clean_months, axis=1)
forecast_base = pd.concat([wide[key_cols].reset_index(drop=True), clean_features], axis=1)

display(forecast_base.head())

## 17. Segmented Robust Forecast

In [ ]:
# ==========================================
# 18. 1. PARAMETER & FUNGSI BANTU
# ==========================================
# Mendefinisikan kolom tren dan fungsi statistik dasar (menghindari error saat data kosong).

non_promo_cols = ["Mar_Clean", "Apr_Clean", "May_Clean", "Jun_Clean", "Aug_Clean"]
recent_non_promo_cols = ["Jun_Clean", "Aug_Clean"]
inactive_3m_cols = ["Jun_Actual", "Jul_Actual", "Aug_Actual"]
forecast_month_labels = [month.strftime("%b") for month in FORECAST_MONTHS]

def nonzero(values):
    # Menyaring array agar hanya menyisakan nilai di atas 0.
    return [float(v) for v in values if float(v) > 0]

def trimmed_mean(values):
    # Menghitung rata-rata dengan membuang nilai ekstrem (terkecil dan terbesar) jika data memadai.
    vals = [float(v) for v in values]
    if len(vals) >= 4:
        vals = sorted(vals)[1:-1]
    return float(np.mean(vals)) if vals else 0.0

# ==========================================
# 18. 1A. SEASONAL INDEX 2025 PER SKU
# ==========================================
# Menghitung seasonal index Sep-Nov dari pola 2025 pada level SKU.
# Seasonal dipakai sebagai pengali konservatif terhadap baseline 2026, bukan sebagai forecast langsung.

seasonal_source = (
    forecast_source[
        forecast_source["Month"].isin(list(SEASONAL_BASE_MONTHS) + list(FORECAST_MONTHS - pd.DateOffset(years=1)))
    ]
    .groupby(["SKU Edit", "Month"], as_index=False)
    .agg(CF=("CF Edit", "sum"))
)

seasonal_wide = seasonal_source.pivot_table(
    index="SKU Edit",
    columns="Month",
    values="CF",
    aggfunc="sum",
    fill_value=0,
).reset_index()

for month in list(SEASONAL_BASE_MONTHS) + list(FORECAST_MONTHS - pd.DateOffset(years=1)):
    if month not in seasonal_wide.columns:
        seasonal_wide[month] = 0

seasonal_reference = seasonal_wide[["SKU Edit"]].copy()
seasonal_reference["Seasonal Base 2025 Avg"] = seasonal_wide[list(SEASONAL_BASE_MONTHS)].replace(0, np.nan).mean(axis=1).fillna(0)

for forecast_month in FORECAST_MONTHS:
    label = forecast_month.strftime("%b")
    reference_month = forecast_month - pd.DateOffset(years=1)
    target_2025 = seasonal_wide[reference_month].fillna(0)
    raw_index = np.where(
        seasonal_reference["Seasonal Base 2025 Avg"].gt(0) & target_2025.gt(0),
        target_2025 / seasonal_reference["Seasonal Base 2025 Avg"],
        1.0,
    )
    clipped_index = pd.Series(raw_index).clip(SEASONAL_MIN_INDEX, SEASONAL_MAX_INDEX)
    seasonal_reference[f"{label} Seasonal Index Raw"] = raw_index
    seasonal_reference[f"{label} Seasonal Index"] = (
        (1 - SEASONAL_BLEND_WEIGHT) * 1.00 + SEASONAL_BLEND_WEIGHT * clipped_index
    )

display(seasonal_reference.head(20))


In [ ]:

# ==========================================
# 18. 2. FUNGSI KLASIFIKASI & FORECAST
# ==========================================
# Menganalisis pola order dan menghitung base forecast inti beserta batas keamanannya (guardrails).

def classify_and_forecast(row):
    non_promo = [row[c] for c in non_promo_cols]
    recent_non_promo = [row[c] for c in recent_non_promo_cols]
    inactive_3m = all(float(row[c]) == 0 for c in inactive_3m_cols)

    nz = nonzero(non_promo)
    active_months = len(nz)
    total_recent = float(sum(non_promo))
    avg = float(np.mean(nz)) if nz else 0.0
    volatility = float(np.std(nz) / avg) if avg > 0 and len(nz) >= 2 else 0.0

    median_recent = float(np.median(nonzero(recent_non_promo))) if nonzero(recent_non_promo) else 0.0
    median_non_promo = float(np.median(nz)) if nz else 0.0
    recent_non_promo_avg = float(0.50 * row["Jun_Clean"] + 0.50 * row["Aug_Clean"])
    trimmed_non_promo = trimmed_mean(non_promo)

    # 1. Menentukan segmen pelanggan dan nilai forecast awal (raw).
    if inactive_3m:
        demand_class, raw_fc = "Inactive 3M", 0.0
    elif total_recent == 0:
        demand_class, raw_fc = "Inactive", 0.0
    elif active_months <= 2:
        demand_class, raw_fc = "New / Sparse", max(median_non_promo, row["Aug_Clean"] * 0.60)
    elif active_months <= 3 or volatility > 1.50:
        demand_class, raw_fc = "Intermittent", median_non_promo
    elif volatility > 0.75:
        demand_class, raw_fc = "Volatile", 0.60 * median_non_promo + 0.40 * trimmed_non_promo
    else:
        demand_class, raw_fc = "Stable", 0.50 * recent_non_promo_avg + 0.50 * median_non_promo

    # 2. Guardrail Atas: Membatasi forecast agar tidak melebihi 120% dari order terbesar historis.
    max_non_promo = max(non_promo) if non_promo else 0.0
    upper_guardrail = max_non_promo * 1.20 if max_non_promo > 0 else raw_fc
    guarded_fc = min(max(raw_fc, 0.0), upper_guardrail)

    # 3. Guardrail Sinyal Lemah: Jika tren bulan Agustus anjlok, forecast tidak boleh lebih tinggi dari median terbaru.
    low_current_signal = median_non_promo > 0 and float(row["Aug_Clean"]) <= median_non_promo * 0.25
    current_month_guardrail = low_current_signal and guarded_fc > median_recent
    base_core = min(guarded_fc, median_recent) if current_month_guardrail else guarded_fc

    return pd.Series({
        "Active Months": active_months,
        "Inactive 3M Rule": inactive_3m,
        "Volatility": volatility,
        "Demand Class": demand_class,
        "Median Recent Non-Promo": median_recent,
        "Median Non-Promo": median_non_promo,
        "Recent Non-Promo Avg": recent_non_promo_avg,
        "Trimmed Mean Non-Promo": trimmed_non_promo,
        "Raw Forecast": raw_fc,
        "Guardrail Cap": upper_guardrail,
        "Current Month Guardrail": current_month_guardrail,
        "Base Forecast Core": base_core,
    })


In [ ]:

# ==========================================
# 18. 3. APPLY METRIK, SEASONAL INDEX, & HITUNG UNIT CASE (UC)
# ==========================================
# Mengeksekusi fungsi forecast, mengalikan seasonal index 2025, dan menambahkan perhitungan UC.

# Menerapkan perhitungan per baris dan menggabungkannya ke dataframe utama.
forecast_metrics = forecast_base.apply(classify_and_forecast, axis=1)
forecast_detail_sep = pd.concat([forecast_base, forecast_metrics], axis=1)

# Menggabungkan seasonal index 2025 per SKU.
seasonal_cols = ["SKU Edit", "Seasonal Base 2025 Avg"]
for month in FORECAST_MONTHS:
    label = month.strftime("%b")
    seasonal_cols += [f"{label} Seasonal Index Raw", f"{label} Seasonal Index"]

forecast_detail_sep = forecast_detail_sep.merge(seasonal_reference[seasonal_cols], on="SKU Edit", how="left")
forecast_detail_sep["Seasonal Base 2025 Avg"] = forecast_detail_sep["Seasonal Base 2025 Avg"].fillna(0)

# Menggabungkan data dengan pengali UC (Unit Case) masing-masing SKU.
forecast_detail_sep = forecast_detail_sep.merge(uc_reference, on=key_cols, how="left")
forecast_detail_sep["UC_Factor"] = forecast_detail_sep["UC_Factor"].fillna(0)

# Membuat forecast Sep-Nov: Base Forecast Core x Seasonal Index 2025 konservatif.
for i, month in enumerate(FORECAST_MONTHS):
    label = month.strftime("%b")
    low_factor = max(0.70, 0.90 - (0.05 * i))
    high_factor = 1.15 + (0.10 * i)

    forecast_detail_sep[f"{label} Seasonal Index Raw"] = forecast_detail_sep[f"{label} Seasonal Index Raw"].fillna(1.0)
    forecast_detail_sep[f"{label} Seasonal Index"] = forecast_detail_sep[f"{label} Seasonal Index"].fillna(1.0)
    forecast_detail_sep[f"{label} Forecast Base"] = forecast_detail_sep["Base Forecast Core"] * forecast_detail_sep[f"{label} Seasonal Index"]
    forecast_detail_sep[f"{label} Forecast Low"] = forecast_detail_sep[f"{label} Forecast Base"] * low_factor
    forecast_detail_sep[f"{label} Forecast High"] = forecast_detail_sep[f"{label} Forecast Base"] * high_factor

    for scenario in ["Low", "Base", "High"]:
        forecast_col = f"{label} Forecast {scenario}"
        forecast_detail_sep[f"{forecast_col} UC"] = forecast_detail_sep[forecast_col] * forecast_detail_sep["UC_Factor"]

# Backward-compatible aliases untuk bagian notebook yang masih membaca nama Sep Forecast.
forecast_detail_sep["Sep Forecast Base"] = forecast_detail_sep["Sep Forecast Base"]
forecast_detail_sep["Sep Forecast Low"] = forecast_detail_sep["Sep Forecast Low"]
forecast_detail_sep["Sep Forecast High"] = forecast_detail_sep["Sep Forecast High"]
forecast_detail_sep["Sep Forecast Base UC"] = forecast_detail_sep["Sep Forecast Base UC"]
forecast_detail_sep["Sep Forecast Low UC"] = forecast_detail_sep["Sep Forecast Low UC"]
forecast_detail_sep["Sep Forecast High UC"] = forecast_detail_sep["Sep Forecast High UC"]

# Mengurutkan tabel berdasarkan forecast dasar Sep (tertinggi ke terendah).
forecast_detail_sep = forecast_detail_sep.sort_values("Sep Forecast Base", ascending=False)

display(forecast_detail_sep.head(30))


## 18. Inactive 3M Customer Matrix

Bagian ini menampilkan `Customer x SKU` yang terkena rule:
`Jun_Actual = 0`, `Jul_Actual = 0`, dan `Aug_Actual = 0`.

Untuk baris ini, forecast September dipaksa menjadi `0`.

In [ ]:
# ==========================================
# 19. 1. FILTERING & DATE PREPARATION
# ==========================================
# Menyaring data Customer x SKU yang tidak aktif 3 bulan terakhir dan menyiapkan rentang bulan.

# Mengambil subset data khusus untuk pelanggan yang terkena aturan Inactive 3M.
inactive_3m_detail = (
    forecast_detail_sep[forecast_detail_sep["Inactive 3M Rule"]]
    .copy()
    .sort_values(["Customer Name", "SKU Edit"])
)

# Menyiapkan daftar bulan untuk sepanjang tahun 2026.
matrix_months = pd.date_range("2026-01-01", "2026-12-01", freq="MS")
matrix_month_labels = [m.strftime("%b_Actual") for m in matrix_months]

# Mengambil data aktual bulanan khusus untuk tahun 2026.
monthly_actual_2026 = monthly_customer_sku[
    monthly_customer_sku["Month"].dt.year.eq(2026)
].copy()

In [ ]:
# ==========================================
# 19. 2. MATRIX CREATION (PIVOT)
# ==========================================
# Membentuk matriks data agar setiap bulan tampil sebagai kolom.

inactive_3m_matrix = (
    inactive_3m_detail[key_cols + ["Inactive 3M Rule", "Demand Class", "Sep Forecast Base"]]
    .merge(monthly_actual_2026, on=key_cols, how="left")
    .pivot_table(
        index=key_cols + ["Inactive 3M Rule", "Demand Class", "Sep Forecast Base"],
        columns="Month",
        values="CF",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)

# Memastikan seluruh bulan (Jan-Des) tersedia sebagai kolom.
for month in matrix_months:
    if month not in inactive_3m_matrix.columns:
        inactive_3m_matrix[month] = 0

# Mengganti nama kolom tanggal dengan format label teks (misal: "Jan_Actual").
inactive_3m_matrix = inactive_3m_matrix[
    key_cols + ["Inactive 3M Rule", "Demand Class", "Sep Forecast Base"] + list(matrix_months)
].rename(columns={month: label for month, label in zip(matrix_months, matrix_month_labels)})

In [ ]:
# ==========================================
# 19. 3. YTD CALCULATION & OUTPUT
# ==========================================
# Menghitung total Actual YTD dan menampilkan ringkasan data.

actual_cols_to_date = [m.strftime("%b_Actual") for m in pd.date_range("2026-01-01", RUNRATE_MONTH, freq="MS")]
inactive_3m_matrix["Actual YTD"] = inactive_3m_matrix[actual_cols_to_date].sum(axis=1)

# Mengurutkan matriks berdasarkan volume YTD terbesar.
inactive_3m_matrix = inactive_3m_matrix.sort_values("Actual YTD", ascending=False)

print("Inactive 3M rows:", len(inactive_3m_detail))
display(inactive_3m_detail.head(30))
display(inactive_3m_matrix.head(50))

In [ ]:
# ==========================================
# 19. 4. HEATMAP VISUALIZATION
# ==========================================
# Membuat heatmap interaktif (Plotly) untuk 50 kombinasi pelanggan x SKU teratas.

if px and len(inactive_3m_matrix) > 0:
    # Mengubah format matriks menjadi memanjang (melt) untuk kebutuhan plot.
    heatmap_source = inactive_3m_matrix.head(50).melt(
        id_vars=["Customer Name", "SKU Edit"],
        value_vars=matrix_month_labels,
        var_name="Month",
        value_name="Actual CF",
    )
    heatmap_source["Customer x SKU"] = heatmap_source["Customer Name"] + " | " + heatmap_source["SKU Edit"]

    # Pivot kembali secara khusus untuk plotting matrix heatmap.
    heatmap_matrix = heatmap_source.pivot_table(
        index="Customer x SKU",
        columns="Month",
        values="Actual CF",
        aggfunc="sum",
        fill_value=0,
    )

    # Merender visualisasi.
    fig = px.imshow(
        heatmap_matrix,
        aspect="auto",
        title="Inactive 3M Customer x SKU Matrix - Actual CF 2026 Top 50 by YTD",
    )
    fig.update_layout(xaxis_title="Month", yaxis_title="Customer x SKU")
    fig.show()

## 19. Forecast Process Visualization

Bagian ini dibuat untuk audit:
- memastikan forecast sudah hanya `MODERN`
- melihat perubahan `Actual` menjadi `Clean`
- melihat dampak run-rate Agustus x1.5
- membedakan Agustus MTD vs Agustus run-rate yang dipakai forecast
- melihat area yang paling banyak terkena promo/outlier cleaning
- melihat distribusi hasil forecast

In [ ]:
# ==========================================
# 20. 1. CHANNEL SCOPE & MONTHLY PROCESS
# ==========================================

scope_months = pd.date_range("2026-01-01", RUNRATE_MONTH, freq="MS")

# Agregasi volume aktual per Channel Group untuk verifikasi scope.
channel_scope = (
    sales_aligned_append[sales_aligned_append["Month"].isin(scope_months)]
    .assign(Channel_Group_Clean=lambda d: d["Channel Group"].map(clean_text))
    .groupby("Channel_Group_Clean", as_index=False)
    .agg(rows=("CF Edit", "size"), cf_edit=("CF Edit", "sum"))
    .sort_values("cf_edit", ascending=False)
)

# Menyusun tabel perbandingan aktual vs baseline bersih per bulan.
monthly_process = []
for month in forecast_input_months:
    mon = month.strftime("%b")
    actual_cf = forecast_detail_sep[f"{mon}_Actual"].sum()
    clean_cf = forecast_detail_sep[f"{mon}_Clean"].sum()

    monthly_process.append({
        "Month": mon,
        "Metric": "Actual Full Month / MTD",
        "CF": actual_cf,
        "Note": "Actual MTD" if month == RUNRATE_MONTH else "Actual full month",
    })

    # Menambahkan metrik run-rate khusus untuk bulan Agustus.
    if month == RUNRATE_MONTH:
        monthly_process.append({
            "Month": mon,
            "Metric": "Actual Run-rate x1.5",
            "CF": forecast_detail_sep["Aug_RunRate"].sum(),
            "Note": "Actual MTD multiplied by fixed 1.5 run-rate",
        })

    monthly_process.append({
        "Month": mon,
        "Metric": "Clean Baseline Used",
        "CF": clean_cf,
        "Note": "Clean MTD x 1.5, used by forecast" if month == RUNRATE_MONTH else "Clean full month, used by forecast",
    })
monthly_process = pd.DataFrame(monthly_process)

In [ ]:
# ==========================================
# 20. 2. DIAGNOSTICS & PROMO CLEANING IMPACT
# ==========================================

# Ringkasan dampak run-rate khusus untuk bulan Agustus.
aug_diagnostic = pd.DataFrame({
    "Metric": [
        "Aug Actual MTD",
        "Aug Actual Run-rate x1.5",
        "Aug Clean Baseline Used",
    ],
    "CF": [
        forecast_detail_sep["Aug_MTD"].sum(),
        forecast_detail_sep["Aug_RunRate"].sum(),
        forecast_detail_sep["Aug_Clean_RunRate"].sum(),
    ],
})

# Menghitung seberapa besar volume yang dipotong (dibersihkan) per SKU.
promo_cleaning_by_sku = (
    forecast_detail_sep
    .groupby("SKU Edit", as_index=False)
    .agg(
        promo_uplift=("Promo_Suspect_Uplift", "sum"),
        promo_months=("Promo_Suspect_Months", "sum"),
        sep_base=("Sep Forecast Base", "sum"),
    )
    .sort_values("promo_uplift", ascending=False)
)

In [ ]:
# ==========================================
# 20. 3. FORECAST AGGREGATIONS
# ==========================================
# Menyiapkan data agregasi forecast untuk divisualisasikan.

viz_by_class = (
    forecast_detail_sep
    .groupby("Demand Class", as_index=False)
    .agg(
        rows=("SKU Edit", "size"),
        sep_base=("Sep Forecast Base", "sum"),
        sep_low=("Sep Forecast Low", "sum"),
        sep_high=("Sep Forecast High", "sum"),
    )
    .sort_values("sep_base", ascending=False)
)

viz_by_branch = (
    forecast_detail_sep
    .groupby("Branch", as_index=False)
    .agg(sep_base=("Sep Forecast Base", "sum"), rows=("SKU Edit", "size"))
    .sort_values("sep_base", ascending=False)
)

viz_by_customer = (
    forecast_detail_sep
    .groupby(["Cust Code", "Customer Name"], as_index=False)
    .agg(sep_base=("Sep Forecast Base", "sum"), rows=("SKU Edit", "size"))
    .sort_values("sep_base", ascending=False)
)

viz_by_sku = (
    forecast_detail_sep
    .groupby("SKU Edit", as_index=False)
    .agg(sep_base=("Sep Forecast Base", "sum"), rows=("Cust Code", "size"))
    .sort_values("sep_base", ascending=False)
)

In [ ]:
# ==========================================
# 20. 4. DISPLAY TABLES & BASIC PLOTS
# ==========================================

display(channel_scope)
display(aug_diagnostic)
display(monthly_process)
display(promo_cleaning_by_sku.head(20))

if px:
    fig = px.bar(
        channel_scope,
        x="Channel_Group_Clean",
        y="cf_edit",
        text="rows",
        title="Forecast Scope Check - Jan to Aug 2026 CF Edit by Channel Group",
    )
    fig.update_layout(xaxis_title="Channel Group", yaxis_title="CF Edit")
    fig.show()

    fig = px.bar(
        monthly_process,
        x="Month",
        y="CF",
        color="Metric",
        barmode="group",
        hover_data=["Note"],
        title="Forecast Baseline Process - Actual, Run-rate, and Clean Baseline",
        category_orders={"Month": [m.strftime("%b") for m in forecast_input_months]},
    )
    fig.update_layout(yaxis_title="CF")
    fig.show()

In [ ]:
# ==========================================
# 20. 5. DETAILED PLOTS & FALLBACK
# ==========================================

if px:
    fig = px.bar(
        promo_cleaning_by_sku.head(20),
        x="SKU Edit",
        y="promo_uplift",
        color="promo_months",
        title="Promo/Outlier Cleaning Impact by SKU - Top 20",
    )
    fig.update_layout(xaxis_title="SKU Edit", yaxis_title="CF Removed from Baseline")
    fig.show()

    fig = px.bar(
        viz_by_class,
        x="Demand Class",
        y="sep_base",
        text="rows",
        title="Sep 2026 Forecast by Demand Class",
    )
    fig.update_layout(yaxis_title="Sep Forecast Base")
    fig.show()

    fig = px.bar(
        viz_by_branch.head(20),
        x="Branch",
        y="sep_base",
        text="rows",
        title="Sep 2026 Forecast by Branch - Top 20",
    )
    fig.update_layout(yaxis_title="Sep Forecast Base")
    fig.show()

    fig = px.bar(
        viz_by_sku.head(20),
        x="SKU Edit",
        y="sep_base",
        text="rows",
        title="Sep 2026 Forecast by SKU - Top 20",
    )
    fig.update_layout(yaxis_title="Sep Forecast Base")
    fig.show()

    fig = px.bar(
        viz_by_customer.head(20),
        x="Customer Name",
        y="sep_base",
        text="rows",
        title="Sep 2026 Forecast by Customer - Top 20",
    )
    fig.update_layout(yaxis_title="Sep Forecast Base")
    fig.show()
else:
    # Menampilkan tabel jika modul px (Plotly) tidak tersedia
    display(viz_by_class)
    display(viz_by_branch.head(20))
    display(viz_by_sku.head(20))
    display(viz_by_customer.head(20))

## 20. Sep-Nov Forecast Summary


In [ ]:

# ==========================================
# 21. 1. OVERALL FORECAST SUMMARY
# ==========================================
# Meringkas hasil akhir metrik forecast secara keseluruhan untuk Sep-Nov.

summary_rows = [
    {"Metric": "Rows", "Value": len(forecast_detail_sep)},
    {"Metric": "Promo Suspect Months", "Value": forecast_detail_sep["Promo_Suspect_Months"].sum()},
    {"Metric": "Promo Suspect Uplift", "Value": forecast_detail_sep["Promo_Suspect_Uplift"].sum()},
]

for month in FORECAST_MONTHS:
    label = month.strftime("%b")
    summary_rows.extend([
        {"Metric": f"{label} Forecast Base", "Value": forecast_detail_sep[f"{label} Forecast Base"].sum()},
        {"Metric": f"{label} Forecast Low", "Value": forecast_detail_sep[f"{label} Forecast Low"].sum()},
        {"Metric": f"{label} Forecast High", "Value": forecast_detail_sep[f"{label} Forecast High"].sum()},
        {"Metric": f"{label} Forecast Base UC", "Value": forecast_detail_sep[f"{label} Forecast Base UC"].sum()},
    ])

summary_sep = pd.DataFrame(summary_rows)
display(summary_sep)


In [ ]:

# ==========================================
# 21. 2. SUMMARY BY DEMAND CLASS & SKU
# ==========================================
# Agregasi data berdasarkan kategori kelas demand dan detail SKU untuk forecast Sep-Nov.

forecast_base_cols = [f"{month.strftime('%b')} Forecast Base" for month in FORECAST_MONTHS]
forecast_low_cols = [f"{month.strftime('%b')} Forecast Low" for month in FORECAST_MONTHS]
forecast_high_cols = [f"{month.strftime('%b')} Forecast High" for month in FORECAST_MONTHS]

summary_by_class_sep = (
    forecast_detail_sep
    .groupby("Demand Class", as_index=False)
    .agg(
        rows=("SKU Edit", "size"),
        promo_months=("Promo_Suspect_Months", "sum"),
        **{col: (col, "sum") for col in forecast_base_cols + forecast_low_cols + forecast_high_cols}
    )
    .sort_values("Sep Forecast Base", ascending=False)
)

summary_by_sku_sep = (
    forecast_detail_sep
    .groupby(["Item Code Edit", "Short Item Description Edit", "SKU Edit"], as_index=False)
    .agg(
        rows=("Cust Code", "size"),
        **{col: (col, "sum") for col in forecast_base_cols + forecast_low_cols + forecast_high_cols}
    )
    .sort_values("Sep Forecast Base", ascending=False)
)

display(summary_by_class_sep)
display(summary_by_sku_sep.head(20))


In [ ]:

# ==========================================
# 21. 3. OUTPUT & VISUALIZATION
# ==========================================
# Menampilkan tabel ringkasan dan membuat grafik bar forecast Base per SKU untuk Sep-Nov.

display(summary_sep)
display(summary_by_class_sep)
display(summary_by_sku_sep.head(30))

if px:
    sku_forecast_long = summary_by_sku_sep.head(20).melt(
        id_vars=["SKU Edit"],
        value_vars=[f"{month.strftime('%b')} Forecast Base" for month in FORECAST_MONTHS],
        var_name="Forecast Month",
        value_name="CF",
    )
    fig = px.bar(
        sku_forecast_long,
        x="SKU Edit",
        y="CF",
        color="Forecast Month",
        barmode="group",
        title="Sep-Nov 2026 Forecast Base by SKU - Top 20 History SKU",
    )
    fig.update_layout(xaxis_title="SKU Edit", yaxis_title="CF")
    fig.show()



## 21. Monthly Forecast Chart

Chart ini menggabungkan history dan forecast dalam satu timeline:
- `Actual Full Month / MTD`: real sales yang tersedia
- `Clean Baseline Used`: baseline setelah promo/outlier cleaning dan Aug run-rate
- `Sep-Nov Forecast`: angka forecast September, Oktober, dan November dengan low-high range


In [ ]:

# ==========================================
# 22. 1. MONTHLY TOTAL & UC PREPARATION
# ==========================================
# Menyiapkan total bulanan actual, clean baseline, dan forecast Sep-Nov dalam CF dan UC.

forecast_rows_cf = []
forecast_rows_uc = []
for month in FORECAST_MONTHS:
    label = month.strftime("%b")
    forecast_rows_cf.extend([
        {"Month_Date": month, "Metric": f"{label} Forecast Low", "CF": forecast_detail_sep[f"{label} Forecast Low"].sum()},
        {"Month_Date": month, "Metric": f"{label} Forecast Base", "CF": forecast_detail_sep[f"{label} Forecast Base"].sum()},
        {"Month_Date": month, "Metric": f"{label} Forecast High", "CF": forecast_detail_sep[f"{label} Forecast High"].sum()},
    ])
    forecast_rows_uc.extend([
        {"Month_Date": month, "Metric": f"{label} Forecast Low", "UC": forecast_detail_sep[f"{label} Forecast Low UC"].sum()},
        {"Month_Date": month, "Metric": f"{label} Forecast Base", "UC": forecast_detail_sep[f"{label} Forecast Base UC"].sum()},
        {"Month_Date": month, "Metric": f"{label} Forecast High", "UC": forecast_detail_sep[f"{label} Forecast High UC"].sum()},
    ])

forecast_monthly_total = pd.concat(
    [
        monthly_process[monthly_process["Metric"].isin(["Actual Full Month / MTD", "Clean Baseline Used"])]
        .assign(Month_Date=lambda d: pd.to_datetime("2026-" + d["Month"] + "-01", format="%Y-%b-%d"))
        [["Month_Date", "Metric", "CF"]],
        pd.DataFrame(forecast_rows_cf),
    ],
    ignore_index=True,
)
forecast_monthly_total["Month Label"] = forecast_monthly_total["Month_Date"].dt.strftime("%b")

forecast_monthly_total.loc[
    (forecast_monthly_total["Metric"] == "Actual Full Month / MTD") &
    (forecast_monthly_total["Month_Date"] == RUNRATE_MONTH),
    "CF",
] = forecast_detail_sep["Aug_RunRate"].sum()

forecast_range_total = pd.DataFrame([
    {
        "Month_Date": month,
        "Month Label": month.strftime("%b"),
        "Forecast Low": forecast_detail_sep[f"{month.strftime('%b')} Forecast Low"].sum(),
        "Forecast Base": forecast_detail_sep[f"{month.strftime('%b')} Forecast Base"].sum(),
        "Forecast High": forecast_detail_sep[f"{month.strftime('%b')} Forecast High"].sum(),
    }
    for month in FORECAST_MONTHS
])

uc_monthly_rows = []
for month in forecast_input_months:
    mon = month.strftime("%b")
    actual_col = "Aug_RunRate" if month == RUNRATE_MONTH else f"{mon}_Actual"

    uc_monthly_rows.append({
        "Month_Date": month,
        "Metric": "Actual Full Month / MTD",
        "UC": float((forecast_detail_sep[actual_col] * forecast_detail_sep["UC_Factor"]).sum()),
    })
    uc_monthly_rows.append({
        "Month_Date": month,
        "Metric": "Clean Baseline Used",
        "UC": float((forecast_detail_sep[f"{mon}_Clean"] * forecast_detail_sep["UC_Factor"]).sum()),
    })

uc_monthly_rows.extend(forecast_rows_uc)

forecast_monthly_total_uc = pd.DataFrame(uc_monthly_rows)
forecast_monthly_total_uc["Month Label"] = forecast_monthly_total_uc["Month_Date"].dt.strftime("%b")

forecast_range_total_uc = pd.DataFrame([
    {
        "Month_Date": month,
        "Month Label": month.strftime("%b"),
        "Forecast Low": forecast_detail_sep[f"{month.strftime('%b')} Forecast Low UC"].sum(),
        "Forecast Base": forecast_detail_sep[f"{month.strftime('%b')} Forecast Base UC"].sum(),
        "Forecast High": forecast_detail_sep[f"{month.strftime('%b')} Forecast High UC"].sum(),
    }
    for month in FORECAST_MONTHS
])


In [ ]:

# ==========================================
# 22. 2. SKU RANKING & MONTHLY FORECAST
# ==========================================
sku_rank_cols = [f"{m.strftime('%b')}_Clean" for m in forecast_input_months]
sku_actual_rank_cols = ["Aug_RunRate" if m == RUNRATE_MONTH else f"{m.strftime('%b')}_Actual" for m in forecast_input_months]

sku_history_rank = (
    forecast_detail_sep
    .assign(
        History_Actual=lambda d: d[sku_actual_rank_cols].sum(axis=1),
        History_Baseline=lambda d: d[sku_rank_cols].sum(axis=1),
    )
    .groupby("SKU Edit", as_index=False)
    .agg(
        history_actual=("History_Actual", "sum"),
        history_baseline=("History_Baseline", "sum"),
        **{f"{m.strftime('%b').lower()}_base": (f"{m.strftime('%b')} Forecast Base", "sum") for m in FORECAST_MONTHS}
    )
    .sort_values("history_actual", ascending=False)
)

sku_monthly_rows = []
top_history_sku = sku_history_rank.head(10)["SKU Edit"].tolist()

def add_monthly_row(rows, sku, month, metric, cf):
    rows.append({
        "SKU Edit": sku,
        "Month_Date": month,
        "Month Label": month.strftime("%b"),
        "Metric": metric,
        "CF": cf,
    })

for sku in top_history_sku:
    sku_rows = forecast_detail_sep[forecast_detail_sep["SKU Edit"].eq(sku)]
    for month in forecast_input_months:
        mon = month.strftime("%b")
        add_monthly_row(sku_monthly_rows, sku, month, "Clean Baseline Used", sku_rows[f"{mon}_Clean"].sum())
    for month in FORECAST_MONTHS:
        label = month.strftime("%b")
        add_monthly_row(sku_monthly_rows, sku, month, f"{label} Forecast Base", sku_rows[f"{label} Forecast Base"].sum())

sku_monthly_forecast = pd.DataFrame(sku_monthly_rows)

display(forecast_monthly_total)
display(forecast_range_total)
display(forecast_monthly_total_uc)
display(forecast_range_total_uc)
display(sku_history_rank.head(10))
display(sku_monthly_forecast.head(40))


In [ ]:
# ==========================================
# 22. 3. MATPLOTLIB CHART HELPERS
# ==========================================
def fmt_k(value, _pos=None):
    value = float(value)
    if abs(value) >= 1000:
        return f"{value / 1000:.0f}k"
    return f"{value:.0f}"

def label_point(ax, x, y, label, color, dx=8, dy=0):
    ax.annotate(
        f"{label}: {y:,.0f}",
        xy=(x, y),
        xytext=(dx, dy),
        textcoords="offset points",
        color=color,
        fontsize=9,
        fontweight="bold",
        va="center",
    )

def label_series_points(ax, data, x_col, y_col, color, dy=8, fontsize=8):
    for _, point in data.iterrows():
        ax.annotate(
            f"{float(point[y_col]):,.0f}",
            xy=(point[x_col], float(point[y_col])),
            xytext=(0, dy),
            textcoords="offset points",
            ha="center",
            va="bottom" if dy >= 0 else "top",
            fontsize=fontsize,
            color=color,
            fontweight="bold",
        )

def label_group_points_no_overlap(ax, data, color_map, x_col="Month_Date", y_col="CF", sku_col="SKU Edit"):
    for month, month_data in data.groupby(x_col):
        month_data = month_data.sort_values(y_col, ascending=False).reset_index(drop=True)
        for rank, point in month_data.iterrows():
            value = float(point[y_col])
            if value == 0:
                continue
            offset = 12 + (rank % 5) * 11
            ax.annotate(
                f"{value:,.0f}",
                xy=(point[x_col], value),
                xytext=(0, offset),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=7,
                color=color_map.get(point[sku_col], "#003049"),
                fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.75),
            )

In [ ]:

# ==========================================
# 22. 4. PLOTTING TOTAL TIMELINE (CF & UC)
# ==========================================
actual_total = forecast_monthly_total[forecast_monthly_total["Metric"].eq("Actual Full Month / MTD")].sort_values("Month_Date")
baseline_total = forecast_monthly_total[forecast_monthly_total["Metric"].eq("Clean Baseline Used")].sort_values("Month_Date")

aug_anchor_month = RUNRATE_MONTH
aug_anchor_value = float(baseline_total.loc[baseline_total["Month_Date"].eq(aug_anchor_month), "CF"].sum())
forecast_base_line = forecast_range_total[["Month_Date", "Forecast Base"]].sort_values("Month_Date")
forecast_low_points = forecast_range_total[["Month_Date", "Forecast Low"]].sort_values("Month_Date")
forecast_high_points = forecast_range_total[["Month_Date", "Forecast High"]].sort_values("Month_Date")

if plt is not None:
    fig, ax = plt.subplots(figsize=(15, 5.5))
    ax.plot(actual_total["Month_Date"], actual_total["CF"], marker="o", linewidth=2.2, color="#4f63ff", label="Actual Full Month / MTD")
    ax.plot(baseline_total["Month_Date"], baseline_total["CF"], marker="o", linewidth=2.2, color="#ef4f3c", label="Clean Baseline Used")
    label_series_points(ax, actual_total, "Month_Date", "CF", "#4f63ff", dy=9)
    label_series_points(ax, baseline_total, "Month_Date", "CF", "#ef4f3c", dy=-14)

    base_months = [aug_anchor_month] + forecast_base_line["Month_Date"].tolist()
    base_values = [aug_anchor_value] + forecast_base_line["Forecast Base"].tolist()
    ax.plot(base_months, base_values, linestyle="--", marker="o", linewidth=2.4, color="#00a676", label="Forecast Base")
    ax.scatter(forecast_low_points["Month_Date"], forecast_low_points["Forecast Low"], s=55, color="#9b7cff", label="Forecast Low", zorder=5)
    ax.scatter(forecast_high_points["Month_Date"], forecast_high_points["Forecast High"], s=55, color="#ff9f43", label="Forecast High", zorder=5)

    for _, row in forecast_range_total.iterrows():
        label_point(ax, row["Month_Date"], row["Forecast Base"], f"{row['Month Label']} Base", "#00a676")
        label_point(ax, row["Month_Date"], row["Forecast Low"], "Low", "#9b7cff", dy=-16)
        label_point(ax, row["Month_Date"], row["Forecast High"], "High", "#ff9f43", dy=16)

    ax.set_title("Monthly Timeline - Actual (Aug Runrate), Clean Baseline, and Sep-Nov Forecast", fontsize=15, fontweight="bold", loc="left")
    ax.set_xlabel("Month")
    ax.set_ylabel("CF")
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_k))
    ax.grid(True, axis="y", alpha=0.25)
    ax.grid(True, axis="x", alpha=0.12)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), frameon=False)
    plt.tight_layout()
    plt.show()

    actual_total_uc = forecast_monthly_total_uc[forecast_monthly_total_uc["Metric"].eq("Actual Full Month / MTD")].sort_values("Month_Date")
    baseline_total_uc = forecast_monthly_total_uc[forecast_monthly_total_uc["Metric"].eq("Clean Baseline Used")].sort_values("Month_Date")
    aug_anchor_value_uc = float(baseline_total_uc.loc[baseline_total_uc["Month_Date"].eq(aug_anchor_month), "UC"].sum())
    forecast_base_line_uc = forecast_range_total_uc[["Month_Date", "Forecast Base"]].sort_values("Month_Date")

    fig, ax = plt.subplots(figsize=(15, 5.5))
    ax.plot(actual_total_uc["Month_Date"], actual_total_uc["UC"], marker="o", linewidth=2.2, color="#4f63ff", label="Actual Full Month / MTD")
    ax.plot(baseline_total_uc["Month_Date"], baseline_total_uc["UC"], marker="o", linewidth=2.2, color="#ef4f3c", label="Clean Baseline Used")
    label_series_points(ax, actual_total_uc, "Month_Date", "UC", "#4f63ff", dy=9)
    label_series_points(ax, baseline_total_uc, "Month_Date", "UC", "#ef4f3c", dy=-14)

    base_months_uc = [aug_anchor_month] + forecast_base_line_uc["Month_Date"].tolist()
    base_values_uc = [aug_anchor_value_uc] + forecast_base_line_uc["Forecast Base"].tolist()
    ax.plot(base_months_uc, base_values_uc, linestyle="--", marker="o", linewidth=2.4, color="#00a676", label="Forecast Base")
    ax.scatter(forecast_range_total_uc["Month_Date"], forecast_range_total_uc["Forecast Low"], s=55, color="#9b7cff", label="Forecast Low", zorder=5)
    ax.scatter(forecast_range_total_uc["Month_Date"], forecast_range_total_uc["Forecast High"], s=55, color="#ff9f43", label="Forecast High", zorder=5)

    for _, row in forecast_range_total_uc.iterrows():
        label_point(ax, row["Month_Date"], row["Forecast Base"], f"{row['Month Label']} Base", "#00a676")
        label_point(ax, row["Month_Date"], row["Forecast Low"], "Low", "#9b7cff", dy=-16)
        label_point(ax, row["Month_Date"], row["Forecast High"], "High", "#ff9f43", dy=16)

    ax.set_title("Monthly Timeline - Actual (Runrate), Clean Baseline, and Sep-Nov Forecast (UC 30L)", fontsize=15, fontweight="bold", loc="left")
    ax.set_xlabel("Month")
    ax.set_ylabel("UC (30L)")
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_k))
    ax.grid(True, axis="y", alpha=0.25)
    ax.grid(True, axis="x", alpha=0.12)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), frameon=False)
    plt.tight_layout()
    plt.show()
else:
    print("Matplotlib is not available in this runtime; chart tables are displayed instead.")


In [ ]:

# ==========================================
# 22. 5. PACK GROUPING & SKU DATA PREP
# ==========================================
def pack_group_from_sku(sku):
    sku = clean_text(sku)
    if "VOLT" in sku or "200 ML" in sku or "200ML" in sku: return "VOLT / SINGLE SERVE"
    if "350ML" in sku or "350 ML" in sku: return "350 ML"
    if "400ML" in sku or "400 ML" in sku or " 400" in sku: return "400 ML"
    if "3LT" in sku or "3L" in sku or "3 LT" in sku: return "3 LITER"
    if "1LT" in sku or "1 LT" in sku or "1000ML" in sku or "1000 ML" in sku: return "1 LITER"
    return "OTHER"

group_rows = []
for sku, sku_rows in forecast_detail_sep.groupby("SKU Edit"):
    group = pack_group_from_sku(sku)
    for month in forecast_input_months:
        mon = month.strftime("%b")
        group_rows.append({
            "Pack Group": group,
            "SKU Edit": sku,
            "Month_Date": month,
            "Month Label": mon,
            "CF": float(sku_rows[f"{mon}_Clean"].sum()),
            "UC": float((sku_rows[f"{mon}_Clean"] * sku_rows["UC_Factor"]).sum()),
            "Point Type": "History",
        })
    for month in FORECAST_MONTHS:
        label = month.strftime("%b")
        group_rows.append({
            "Pack Group": group,
            "SKU Edit": sku,
            "Month_Date": month,
            "Month Label": label,
            "CF": float(sku_rows[f"{label} Forecast Base"].sum()),
            "UC": float(sku_rows[f"{label} Forecast Base UC"].sum()),
            "Point Type": "Forecast",
        })

sku_group_monthly = pd.DataFrame(group_rows)
group_order = (
    sku_group_monthly[sku_group_monthly["Point Type"].eq("Forecast")]
    .groupby("Pack Group", as_index=False)
    .agg(sep_nov_base=("CF", "sum"), sku_count=("SKU Edit", "nunique"))
    .query("sep_nov_base > 0 and `Pack Group` != 'OTHER'")
    .sort_values("sep_nov_base", ascending=False)
)

group_order_uc = (
    sku_group_monthly[sku_group_monthly["Point Type"].eq("Forecast")]
    .groupby("Pack Group", as_index=False)
    .agg(sep_nov_base_uc=("UC", "sum"), sku_count=("SKU Edit", "nunique"))
    .query("sep_nov_base_uc > 0 and `Pack Group` != 'OTHER'")
    .sort_values("sep_nov_base_uc", ascending=False)
)

display(group_order)
display(group_order_uc)
display(sku_group_monthly.head(40))


In [ ]:

# ==========================================
# 22. 6. PLOTTING SKU LEVEL CHARTS
# ==========================================
palette = ["#005f73", "#2ca25f", "#f59e0b", "#6d5dfc", "#ef476f", "#118ab2"]

if plt is not None:
    for _, group_row in group_order.iterrows():
        group = group_row["Pack Group"]
        group_data = sku_group_monthly[sku_group_monthly["Pack Group"].eq(group)].copy()
        top_group_skus = (
            group_data[group_data["Point Type"].eq("Forecast")]
            .groupby("SKU Edit", as_index=False)
            .agg(total_forecast=("CF", "sum"))
            .sort_values("total_forecast", ascending=False)
            .head(5)["SKU Edit"]
            .tolist()
        )
        chart_data = group_data[group_data["SKU Edit"].isin(top_group_skus)]
        color_map = {sku: palette[i % len(palette)] for i, sku in enumerate(top_group_skus)}

        fig, ax = plt.subplots(figsize=(15, 4.8))
        for i, sku in enumerate(top_group_skus):
            sku_data = chart_data[chart_data["SKU Edit"].eq(sku)].sort_values("Month_Date")
            hist = sku_data[sku_data["Point Type"].eq("History")]
            fcst = sku_data[sku_data["Point Type"].eq("Forecast")]
            color = color_map[sku]

            ax.plot(hist["Month_Date"], hist["CF"], marker="o", linewidth=2.3, color=color, label=sku)
            if not hist.empty and not fcst.empty:
                bridge_months = [hist["Month_Date"].max()] + fcst["Month_Date"].tolist()
                bridge_values = [float(hist.loc[hist["Month_Date"].idxmax(), "CF"])] + fcst["CF"].tolist()
                ax.plot(bridge_months, bridge_values, linestyle="--", marker="o", linewidth=2.3, color=color)

        label_group_points_no_overlap(ax, chart_data, color_map)

        ax.set_title(f"{group} - {len(top_group_skus)} SKU", fontsize=14, fontweight="bold", loc="left")
        ax.set_ylabel("CF")
        ax.yaxis.set_major_formatter(FuncFormatter(fmt_k))
        ax.margins(y=0.18)
        ax.grid(True, axis="y", alpha=0.25)
        ax.grid(True, axis="x", alpha=0.10)
        ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False, fontsize=9)
        plt.tight_layout()
        plt.show()

    for _, group_row in group_order_uc.iterrows():
        group = group_row["Pack Group"]
        group_data = sku_group_monthly[sku_group_monthly["Pack Group"].eq(group)].copy()
        top_group_skus = (
            group_data[group_data["Point Type"].eq("Forecast")]
            .groupby("SKU Edit", as_index=False)
            .agg(total_forecast=("UC", "sum"))
            .sort_values("total_forecast", ascending=False)
            .head(5)["SKU Edit"]
            .tolist()
        )
        chart_data = group_data[group_data["SKU Edit"].isin(top_group_skus)]
        color_map = {sku: palette[i % len(palette)] for i, sku in enumerate(top_group_skus)}

        fig, ax = plt.subplots(figsize=(15, 4.8))
        for i, sku in enumerate(top_group_skus):
            sku_data = chart_data[chart_data["SKU Edit"].eq(sku)].sort_values("Month_Date")
            hist = sku_data[sku_data["Point Type"].eq("History")]
            fcst = sku_data[sku_data["Point Type"].eq("Forecast")]
            color = color_map[sku]

            ax.plot(hist["Month_Date"], hist["UC"], marker="o", linewidth=2.3, color=color, label=sku)
            if not hist.empty and not fcst.empty:
                bridge_months = [hist["Month_Date"].max()] + fcst["Month_Date"].tolist()
                bridge_values = [float(hist.loc[hist["Month_Date"].idxmax(), "UC"])] + fcst["UC"].tolist()
                ax.plot(bridge_months, bridge_values, linestyle="--", marker="o", linewidth=2.3, color=color)

        label_group_points_no_overlap(ax, chart_data, color_map, y_col="UC")

        ax.set_title(f"{group} - {len(top_group_skus)} SKU (UC 30L)", fontsize=14, fontweight="bold", loc="left")
        ax.set_ylabel("UC (30L)")
        ax.yaxis.set_major_formatter(FuncFormatter(fmt_k))
        ax.margins(y=0.18)
        ax.grid(True, axis="y", alpha=0.25)
        ax.grid(True, axis="x", alpha=0.10)
        ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False, fontsize=9)
        plt.tight_layout()
        plt.show()


## 22. Export Forecast CSV

File CSV akan disimpan di folder `forecast_output` pada folder yang sama dengan file dashboard di Google Drive.

In [ ]:
# ==========================================
# 23. 1. SETUP & DIRECTORY CREATION
# ==========================================
# Menyiapkan direktori penyimpanan file ekspor di Google Drive dan daftar SKU yang dikecualikan.

EXPORT_DIR = DRIVE_SALES_FILE.parent / "forecast_output"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
forecast_label = f"{FORECAST_MONTHS[0].strftime('%Y_%m')}_to_{FORECAST_MONTHS[-1].strftime('%Y_%m')}"

# SKU yang dikecualikan sesuai permintaan pengguna.
excluded_skus = ['507563', '522635', '522636', '523229', '523415']


In [ ]:
# ==========================================
# 23. 2. PREPARASI DATA TRANSAKSI (FILE 01)
# ==========================================
# Menerapkan filter eksklusi SKU pada data sumber transaksi aktual.

order_clean_adjusted = forecast_source[
    ~forecast_source['Item Code Edit'].isin(excluded_skus)
][[
    "Source Sheet", "Year", "Month No", "Date",
    "Channel Group", "Original Customer Name", "Customer Name", "Original Branch", "Branch", "Channel", "Cust Code",
    "Short Item Description Real", "Item Code Real",
    "Short Item Description Edit", "Item Code Edit",
    "Format", "Box Content", "Reference Format", "Reference Box Content",
    "CF Real", "CF Edit", "CF Adjustment",
    "Group Key", "SKU Real", "SKU Edit", "Real != Edit",
]].copy()

In [ ]:
# ==========================================
# 23. 3. SKELETON FULL MATRIX (CROSS JOIN)
# ==========================================
# Membuat kombinasi lengkap (cross join) antara daftar customer dan SKU aktif.

cust_list = forecast_source[['Cust Code', 'Customer Name', 'Branch', 'Channel Group']].drop_duplicates('Cust Code')

sku_list = forecast_source[
    ~forecast_source['Item Code Edit'].isin(excluded_skus)
][['Item Code Edit', 'Short Item Description Edit', 'SKU Edit']].drop_duplicates('SKU Edit')

skeleton = cust_list.assign(key=1).merge(sku_list.assign(key=1), on='key').drop('key', axis=1)

In [ ]:
# ==========================================
# 23. 4. PIVOT DATA AKTUAL 2026
# ==========================================
# Membentuk pivot table bulanan untuk seluruh transaksi aktual tahun 2026.

export_months = pd.date_range("2026-01-01", "2026-12-01", freq="MS")
export_month_labels = [m.strftime("%b_Actual") for m in export_months]

actual_2026_pivot = (
    monthly_customer_sku[
        monthly_customer_sku["Month"].dt.year.eq(2026) &
        ~monthly_customer_sku['Item Code Edit'].isin(excluded_skus)
    ]
    .pivot_table(index=key_cols, columns="Month", values="CF", aggfunc="sum", fill_value=0)
    .reset_index()
)

for month in export_months:
    if month not in actual_2026_pivot.columns:
        actual_2026_pivot[month] = 0

actual_2026_pivot = actual_2026_pivot[key_cols + list(export_months)].rename(
    columns={month: label for month, label in zip(export_months, export_month_labels)}
)

In [ ]:

# ==========================================
# 23. 5. GABUNGKAN SKELETON DENGAN FORECAST
# ==========================================
# Menggabungkan rangka matriks dengan data kalkulasi forecast dan pivot bulanan.

forecast_calc_cols = [
    "Mar_Clean", "Apr_Clean", "May_Clean", "Jun_Clean", "Aug_Clean",
    "Aug_MTD", "Aug_RunRate", "Aug_Clean_RunRate",
    "Promo_Suspect_Months", "Promo_Suspect_Uplift",
    "Active Months", "Inactive 3M Rule", "Volatility", "Demand Class",
    "Base Forecast Core", "Current Month Guardrail",
    "Seasonal Base 2025 Avg", "UC_Factor",
]

for month in FORECAST_MONTHS:
    label = month.strftime("%b")
    forecast_calc_cols.extend([
        f"{label} Seasonal Index Raw",
        f"{label} Seasonal Index",
        f"{label} Forecast Low",
        f"{label} Forecast Base",
        f"{label} Forecast High",
        f"{label} Forecast Low UC",
        f"{label} Forecast Base UC",
        f"{label} Forecast High UC",
    ])

forecast_detail_filtered = forecast_detail_sep[
    ~forecast_detail_sep['Item Code Edit'].isin(excluded_skus)
].copy()

full_matrix_data = skeleton.merge(forecast_detail_filtered[key_cols + forecast_calc_cols], on=key_cols, how='left')
full_matrix_data = full_matrix_data.merge(actual_2026_pivot[key_cols + export_month_labels], on=key_cols, how='left')

# Penanganan nilai kosong (NaN) setelah penggabungan matriks.
full_matrix_data['Demand Class'] = full_matrix_data['Demand Class'].fillna("No History")
full_matrix_data[export_month_labels] = full_matrix_data[export_month_labels].fillna(0)
numeric_cols = [c for c in full_matrix_data.columns if 'Forecast' in c or 'Clean' in c or 'Uplift' in c or 'UC' in c or 'Seasonal' in c]
full_matrix_data[numeric_cols] = full_matrix_data[numeric_cols].fillna(0)


In [ ]:

# ==========================================
# 23. 6. EKSPOR KE FILE CSV
# ==========================================
# Memisahkan dataframe final menjadi 3 file terpisah dan menyimpannya ke direktori tujuan.

pivot_full_export = full_matrix_data[key_cols + export_month_labels + forecast_calc_cols].copy()

forecast_month_cols = []
for month in FORECAST_MONTHS:
    label = month.strftime("%b")
    forecast_month_cols.extend([
        f"{label} Seasonal Index",
        f"{label} Forecast Low",
        f"{label} Forecast Base",
        f"{label} Forecast High",
        f"{label} Forecast Low UC",
        f"{label} Forecast Base UC",
        f"{label} Forecast High UC",
    ])

forecast_sep_full = full_matrix_data[[
    "Channel Group", "Branch", "Cust Code", "Customer Name",
    "Item Code Edit", "Short Item Description Edit", "SKU Edit",
    "Demand Class", "Inactive 3M Rule", "UC_Factor",
    "Base Forecast Core", "Seasonal Base 2025 Avg",
    *forecast_month_cols,
]].copy()
forecast_sep_full.insert(0, "Forecast Period", f"{FORECAST_MONTHS[0].strftime('%Y-%m')} to {FORECAST_MONTHS[-1].strftime('%Y-%m')}")

export_files = {
    f"01_order_clean_adjusted_{forecast_label}_modern.csv": order_clean_adjusted,
    f"02_pivot_customer_sku_full_matrix_{forecast_label}_modern.csv": pivot_full_export,
    f"03_forecast_sep_nov_full_matrix_{forecast_label}_modern.csv": forecast_sep_full,
}

exported_paths = []
for filename, dataframe in export_files.items():
    output_path = EXPORT_DIR / filename
    dataframe.to_csv(output_path, index=False, encoding="utf-8-sig")
    exported_paths.append(str(output_path))

print("Export folder:", EXPORT_DIR)
print(f"Rows in Full Matrix (Excluding {len(excluded_skus)} SKU types): {len(full_matrix_data)}")
print("Exported CSV files:")
for path in exported_paths:
    print("-", path)


### Audit Khusus: Bali & Pekanbaru
Sel ini bertujuan untuk melacak apakah volume historis dari area Bali dan Pekanbaru tertangkap dalam perhitungan, meskipun ada perubahan nama branch.

In [ ]:
# ==========================================
# 24. 1. AREA AUDIT BALI & PEKANBARU
# ==========================================
# Memeriksa volume transaksi dan status customer pada area ekspansi Bali dan Pekanbaru.

area_keywords = ['BALI', 'PEKANBARU']

# Mencari data transaksi pada branch atau customer yang mengandung kata kunci area.
area_audit = sales_aligned_append[
    sales_aligned_append['Branch'].str.contains('|'.join(area_keywords), case=False, na=False) |
    sales_aligned_append['Customer Name'].str.contains('|'.join(area_keywords), case=False, na=False)
].copy()

# Membuat ringkasan volume (CF Edit) per Branch dan Bulan.
area_summary = area_audit.groupby(['Branch', 'Month']).agg(CF=('CF Edit', 'sum')).reset_index()
area_summary = area_summary.pivot(index='Branch', columns='Month', values='CF').fillna(0)

print("Ringkasan Volume di Branch terkait Bali/Pekanbaru:")
display(area_summary)